# Closing the landmark→latest gap

**The question.** A seed's citation graph shows *Field Landmarks* (the all-time
most-cited citers, at any year) and *Latest Publications* (recent citers, filled
one `cited_by_count` query per year over a **fixed** span — `latest_band_years=5`,
so 2020-2024 today). For an *old* seed whose landmark cluster tails off years
before 2020, the timeline shows a dead stretch between the last landmark and the
first band. Where should the bands *start*, per seed, so that gap closes?

This notebook is the argument; the shipped, re-runnable fit lives in
`ml_pipelines/latest_gap/` and the served rule in
`atlas.services.graph.bands`.

In [1]:
import csv
import math
from collections import Counter

import numpy as np

# The corpus lives with the pipeline, under the src-layout root (single copy):
# each seed's shipped-landmark publication years (citation-ranked, budget-trimmed)
# — exactly the pool the build would ship as Field Landmarks.
CORPUS = "../../src/ml_pipelines/latest_gap/corpus.csv"
FIXED = 5  # today's latest_band_years

rows = []
with open(CORPUS, newline="") as handle:
    for row in csv.DictReader(handle):
        rows.append({
            "label": row["label"],
            "years": [int(year) for year in row["citer_years"].split()],
            "lm_max": int(row["landmark_max_year"]),
            "is_worked_example": int(row["is_worked_example"]),
        })
print(f"{len(rows)} seeds")

64 seeds


## 1. The gap is real, and it's an *old-seed* phenomenon

For each seed, the fixed bands cover `[lm_max-4, lm_max]`. The **visible gap** is
the longest run of empty years between the landmark cluster and that band region
(a year is "populated" if it holds a landmark or falls in the band range).

In [2]:
def visible_gap(years, band_start, lm_max):
    """The longest run of empty years between the landmark cluster and the band region.

    A year counts as populated if it holds a landmark or falls in the band
    range ``[band_start, lm_max]``; the gap is the longest stretch of
    unpopulated years within the seed's overall span.
    """
    populated = set(years) | set(range(band_start, lm_max + 1))
    lo, hi = min(populated), max(populated)
    longest = current = 0
    for year in range(lo, hi + 1):
        current = 0 if year in populated else current + 1
        longest = max(longest, current)
    return longest

for row in rows:
    fixed_start = row["lm_max"] - FIXED + 1
    row["age"] = 2026 - min(row["years"] + [2026])  # rough: oldest citer as age proxy
    row["gap_now"] = visible_gap(row["years"], fixed_start, row["lm_max"])

gapped = [row for row in rows if row["gap_now"] >= 3]
print(f"{len(gapped)} seeds have a visible gap >= 3 years under the fixed span")
# The gapped seeds skew old (their landmark cluster's most-recent year is early).
recent_edges = sorted(max(row["years"]) for row in gapped)
print("most-recent landmark year among gapped seeds:", recent_edges)

10 seeds have a visible gap >= 3 years under the fixed span
most-recent landmark year among gapped seeds: [2015, 2015, 2021, 2021, 2021, 2022, 2022, 2023, 2023, 2023]


## 2. Can seed features predict the fix? (No.)

The sibling `cite_budget` model predicts the landmark *budget* from seed age +
log-citations. The natural first idea is to reuse that: predict the needed band
**span** from the same features. It **fails** — the boundary depends on the
*shape* of each seed's landmark distribution, which age/citations don't capture.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score


def tail_edge(years, threshold, lm_max):
    """The most recent year whose absolute landmark count reaches ``threshold``.

    Scans back from ``lm_max``; falls back to the earliest citer year when no
    year clears the bar. This is the raw-count density edge used to derive the
    span label below.
    """
    counts = Counter(years)
    for year in range(lm_max, min(years) - 1, -1):
        if counts[year] >= threshold:
            return year
    return min(years)

# Label: band span the density edge wants (floored at the fixed 5).
spans = np.array([max(row["lm_max"] - tail_edge(row["years"], 2, row["lm_max"]), FIXED)
                  for row in rows])
# Features: age + log-citations (as cite_budget uses). Age proxied from lm_max.
features = np.array([[2026 - (row["lm_max"] - len(row["years"]) // 10),
                      math.log10(len(row["years"]) + 1)]
                     for row in rows])
model = LinearRegression().fit(features, spans)
cv = cross_val_score(model, features, spans, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
print(f"feature-regression CV R2 = {cv:.3f}  (negative: features don't predict the boundary)")

feature-regression CV R2 = -0.150  (negative: features don't predict the boundary)


So the model must operate on the **distribution itself**, which the build
already has in hand (landmarks are fetched before the bands). What we fit is the
rule's parameters, not feature coefficients.

## 3. A quantile is the wrong detector — it's dragged by the old bulk

The obvious "recent edge" statistic is a high quantile of the landmark years. But
a quantile is **mass-based**: when a seed has a large *old* bulk, that bulk drags
the percentile years before the cluster's visible edge. Compare the 0.85 quantile
to a **density tail edge** — the most recent year whose per-year count is still
≥ `tau` of the peak year's count — on a few old seeds:

In [4]:
def quantile_year(years, fraction):
    """The year at quantile ``fraction`` of the sorted landmark years.

    The mass-based recent-edge statistic — shown here to fail against the
    density edge, since a large old bulk drags the quantile before the
    cluster's visible edge.
    """
    ordered = sorted(years)
    return ordered[min(int(fraction * len(ordered)), len(ordered) - 1)]

def tail_edge(years, tau):
    """The most recent year whose per-year count is still at least ``tau`` of the peak.

    The density tail edge the shipped rule uses: ``tau`` is a fraction of the
    busiest year's landmark count, so the edge tracks where the cluster
    actually thins rather than where its mass sits.
    """
    counts = Counter(years)
    threshold = max(tau * max(counts.values()), 1.0)
    for year in range(max(years), min(years) - 1, -1):
        if counts[year] >= threshold:
            return year
    return min(years)

print(f"{'seed':<26}{'span':>12}{'q0.85':>7}{'tail(0.25)':>11}")
for label in ["Hawking Radiation", "Theory of the firm: Managerial behavior, agency costs and ownership structure",
              "A Rapid and Sensitive Method for the Quantitation of Microgram Quantities of Protein Utilizing the Principle of Protein-Dye Binding",
              "QMIX"]:
    row = next((candidate for candidate in rows if candidate["label"] == label), None)
    if not row:
        continue
    years = row["years"]
    print(f"{label[:24]:<26}{min(years):>6}-{max(years):<5}"
          f"{quantile_year(years, 0.85):>7}{tail_edge(years, 0.25):>11}")

seed                              span  q0.85 tail(0.25)
Hawking Radiation           1975-2023    2013       2020
Theory of the firm: Mana    1977-2015    2008       2014
A Rapid and Sensitive Me    1978-2015    2005       2015
QMIX                        2018-2024    2022       2024


Hawking's cluster stays dense until ~2020 (the visible cliff on the
timeline), but its 0.85 quantile sits at **2013** — seven years too early,
because 160 landmarks pile up in 2000–2019. The **tail edge lands at 2020**,
matching the eye. So the quantile systematically *undershoots* and the `max_span`
clamp was quietly doing all the real work. The density edge tracks where the
count actually falls off.

## 4. Fitting `tau` — gap closure is flat, so fit on robustness

The rule: **band start = the density tail edge, floored by a `max_span` cap** (no
only-widen clamp — a young seed starts at its own recent edge). Sweeping `tau`
shows gap closure barely moves (the cap dominates), so `tau` isn't a
gap knob — its real effect is **robustness to OpenAlex's misdated years**, the
arc's recurring hazard. A higher threshold needs more same-year citers to shift
the edge:

In [5]:
def moved_by_misdate(tau):
    """How many seeds' density tail edges shift when two misdated citers are appended.

    Appends two citers dated two years past the newest (OpenAlex's recurring
    misdating hazard) and counts the seeds whose ``tail_edge`` moves — the
    robustness measure ``tau`` is fit on.
    """
    moved = 0
    for row in rows:
        years = row["years"]
        outlier = max(years) + 2
        if tail_edge(years, tau) != tail_edge(years + [outlier, outlier], tau):
            moved += 1
    return moved

print(f"{'tau':>5}{'seeds moved by a 2-citer misdate':>34}")
for tau in (0.10, 0.15, 0.20, 0.25, 0.30):
    print(f"{tau:>5}{moved_by_misdate(tau):>34} / {len(rows)}")

  tau  seeds moved by a 2-citer misdate
  0.1                                58 / 64
 0.15                                38 / 64
  0.2                                10 / 64
 0.25                                 1 / 64
  0.3                                 0 / 64


So `tau` is fit as the **smallest threshold robust to a two-citer misdate**
(≤ 5% of seeds move) — `tau = 0.25` (1/64 movable), the cheapest robust choice.
`max_span = 7` is a cost cap (Patrick's pick): worst case `7 + 2 = 9` band queries
(the `+2` are the two latest-only years always banded up to today).

In [6]:
# The four worked-example seeds under the shipped rule (tau=0.25, max_span=7).
TAU, MAX_SPAN, CUR = 0.25, 7, 2026
for row in rows:
    if row["is_worked_example"]:
        years, lm = row["years"], row["lm_max"]
        start = max(tail_edge(years, TAU), lm - MAX_SPAN + 1)
        print(f"{row['label']:<28} landmarks {min(years)}-{max(years)}  "
              f"band_start {start}  ({CUR - start + 1} bands)")

Hawking Radiation            landmarks 1975-2023  band_start 2020  (7 bands)
DQN                          landmarks 2015-2021  band_start 2021  (6 bands)
QMIX                         landmarks 2018-2024  band_start 2024  (3 bands)
Attention Is All You Need    landmarks 2017-2022  band_start 2022  (5 bands)


## Conclusion

The landmark→latest gap is closed by starting the per-year bands at the **density
tail edge** of the shipped landmarks — the most recent still-dense year — floored
by a `max_span` cost cap, with **no only-widen clamp**. A quantile looked right
but undershoots (dragged by the old bulk); the density edge matches the visible
cluster. `tau` is fit on misdate-robustness rather than gap closure (which is flat
across it). Old seeds (Hawking → 2020) widen back to meet the cluster; young seeds
(QMIX → 2024, three bands) get a tight recent frontier. Seeds whose "gap" is an
*internal* landmark hole are correctly left alone — banding can't fill a hole
inside the cluster.

Productionized in `ml_pipelines/latest_gap/` (fit → `model.joblib` beside the
trainer), served by `atlas.services.graph.bands`, and applied in
`integrations/openalex/traversal.citation_relations`. The vocabulary here — tail
edge, `tau`, `max_span`, band start — is defined in
[`docs/landmark-vocabulary.md`](../../docs/landmark-vocabulary.md).